# Guideline → policy matching: three ways to feed an LLM

**Task.** Given a guideline and a set of payer policies, find which policy criteria each guideline criterion bears on and say how they differ. This is the unspecified "matching" step in `docs/BACKEND_SPEC.md` (`criterion_match`).

| Arm | Representation | Who finds matches and judges them |
|---|---|---|
| **A. plain text** | `{document_name, text}` | the LLM, reading the guideline and **all** policy documents |
| **B. criteria dump** | atomized criteria JSON, `.text` only (today's schema) | the LLM, reading **all** criteria |
| **C. ontology** | one structured record per criterion, terms mapped to shared Concepts | **code**: Concept lookup, then fixed slot-by-slot rules. The LLM only extracts each record once (offline) and writes the final sentence |
| **C′. ontology, no normalization** | same records, Concepts are just the lowercased words | same code, to measure what normalization buys |

A→B isolates the value of atomizing. B→C isolates the value of structure plus deterministic matching. C→C′ isolates the value of normalizing terms with real vocabularies (RxNorm, SNOMED CT, RxClass; see `docs/INTERVENTION_NORMALIZATION.md`).

## The ontology (arm C)

Every criterion, guideline or policy, is one sentence: **[stance] [intervention] for [who], limited by [delivery]**.

- **stance**: `for` or `against`. Guideline words (recommends, suggests, recommends against) and policy words (covers, not covered, experimental) both map here, so the two sides compare directly.
- **intervention**: what is done (drug, drug class, vaccine, procedure, therapy, test, device). Points at a **Concept**.
- **who**: facts about the patient, each a Concept with an optional range (`age 50–80`, `BMI ≥ 35`, `has type 2 diabetes`), joined by `all_of` / `any_of`.
- **delivery**: how often, where, by whom, prior authorization.
- **Concept**: shared vocabulary with codes and broader/narrower links, filled by **lookup, not by the LLM**.

**Comparison rules** (`compare` below), policy relative to guideline:
1. Pair a guideline record with every policy record whose intervention is the same Concept or a broader/narrower one.
2. **who** is a set of patients: `BMI ≥ 40` is a subset of `BMI ≥ 35`; an extra `all_of` fact makes the set smaller. **delivery**: a limit only the policy has makes it stricter.
3. Verdict: opposite stances → `conflict`; policy who a subset or extra limits → `policy_stricter`; superset → `policy_looser`; equal → `aligned`; no policy record → `uncovered`.

**Choices I made where the spec is silent** (change them in `compare`):
- A policy record whose **who** is *disjoint* from the guideline's is not paired (a 76–85 guideline criterion is not "contradicted" by a 50–75 policy criterion). A guideline criterion left with no pair is `uncovered`.
- Same stance with *overlapping but incomparable* who is dropped as undecidable rather than guessed.
- A policy naming a *narrower* intervention (one statin vs the class) counts as stricter; *broader* as looser.

**Reading the results.** The fixtures are tiny, synthetic and hand-labelled (not clinical advice), each arm runs once. Treat this as a smoke test of the design, not a benchmark. Swap `GUIDELINES` / `POLICIES` / `GOLD` for real atomized rows to scale it up.

Run: `uv run --with jupyter --with pandas --with httpx jupyter lab` from the repo root, with `OPENAI_API_KEY` set (optionally `OPENAI_MODEL`). Arm C's normalization calls public keyless NLM / NCI / HL7 services. LLM responses and normalization results are cached in `notebooks/.cache/`, so re-runs are free.

In [ ]:
import hashlib
import json
import os
import re
import sys
from dataclasses import dataclass
from difflib import get_close_matches
from pathlib import Path
from typing import Literal, Optional, TypeVar

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

sys.path.insert(0, str(Path.cwd()))  # the notebook lives next to intervention_normalizer.py
from intervention_normalizer import Concept, Kind, Normalized, Normalizer  # noqa: E402

MODEL = os.environ.get("OPENAI_MODEL", "gpt-6-luna")  # same default as policy_radar/config.py
CACHE_DIR = Path(".cache/rag_compare")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

M = TypeVar("M", bound=BaseModel)
LEDGER: list[dict] = []  # one row per LLM call: arm, tokens, cached?
_client: Optional[OpenAI] = None


def ask(arm: str, schema: type[M], instructions: str, payload: str) -> M:
    """One structured LLM call, cached on disk. Token usage is recorded per arm."""
    global _client
    key = hashlib.sha256(
        json.dumps([MODEL, schema.__name__, instructions, payload]).encode()
    ).hexdigest()
    path = CACHE_DIR / f"{key}.json"
    if path.exists():
        hit = json.loads(path.read_text())
        LEDGER.append({"arm": arm, "in": hit["in"], "out": hit["out"], "cached": True})
        return schema.model_validate_json(hit["parsed"])
    if _client is None:
        _client = OpenAI()  # raises a clear error if OPENAI_API_KEY is unset
    resp = _client.responses.parse(
        model=MODEL, instructions=instructions, input=payload, text_format=schema
    )
    if resp.output_parsed is None:
        raise ValueError(f"no parsable output for arm {arm}")
    usage = {"in": resp.usage.input_tokens, "out": resp.usage.output_tokens}
    path.write_text(json.dumps({**usage, "parsed": resp.output_parsed.model_dump_json()}))
    LEDGER.append({"arm": arm, **usage, "cached": False})
    return resp.output_parsed

## 1. Fixture data

Criteria mirror `GuidelineCriterion` / `PolicyCriterion` (`criterion_id = "{parent_id}:{position}"`). Plain-text documents are rendered from the same criteria plus boilerplate, so all arms see the same facts packaged differently. Distractors (CT colonography, hepatitis C, knee arthroscopy) punish over-matching.

`GOLD_DETAIL` is the label a careful reviewer would give each pair; `"-"` means the guideline criterion has no policy counterpart. `GOLD` is its binary form (anything but `aligned` needs review).

In [ ]:
def crit(parent: str, texts: list[str]) -> list[dict]:
    return [{"criterion_id": f"{parent}:{i}", "parent_id": parent, "text": t} for i, t in enumerate(texts, 1)]


GUIDELINES = {
    "G-CRC": {
        "name": "USPSTF Colorectal Cancer Screening (fixture)",
        "criteria": crit("G-CRC", [
            "Average-risk adults aged 45 to 49 years should be screened for colorectal cancer.",
            "Average-risk adults aged 50 to 75 years should be screened for colorectal cancer.",
            "Average-risk adults aged 45 to 75 years may be screened with a fecal immunochemical test (FIT) once every year.",
            "Average-risk adults aged 45 to 75 years may be screened with colonoscopy once every 10 years.",
            "Adults aged 76 to 85 years should be offered colorectal cancer screening selectively, based on overall health.",
        ]),
    },
    "G-HBV": {
        "name": "ACIP Adult Hepatitis B Vaccination (fixture)",
        "criteria": crit("G-HBV", [
            "Adults aged 19 to 59 years should receive hepatitis B vaccination.",
            "Adults aged 60 years and older with risk factors for hepatitis B infection should receive hepatitis B vaccination.",
        ]),
    },
    "G-LIP": {
        "name": "ACC/AHA Cholesterol Management (fixture)",
        "criteria": crit("G-LIP", [
            "Adults with LDL cholesterol of 190 mg/dL or higher should receive statin therapy.",
        ]),
    },
}

POLICIES = {
    "0101": {
        "name": "CPB 0101 Colorectal Cancer Screening",
        "criteria": crit("0101", [
            "Colorectal cancer screening is medically necessary for average-risk members aged 50 to 75 years.",
            "Screening colonoscopy is covered once every 10 years for average-risk members aged 50 to 75 years.",
            "Fecal immunochemical test (FIT) is covered once every year for average-risk members aged 50 to 75 years.",
            "Colorectal cancer screening is not medically necessary for members older than 75 years.",
        ]),
    },
    "0102": {
        "name": "CPB 0102 CT Colonography",
        "criteria": crit("0102", [
            "CT colonography is considered experimental and investigational for colorectal cancer screening.",
            "CT colonography is medically necessary for members with an incomplete diagnostic colonoscopy.",
        ]),
    },
    "0201": {
        "name": "CPB 0201 Hepatitis B Vaccine",
        "criteria": crit("0201", [
            "Hepatitis B vaccination is covered for adults aged 19 to 59 years, with prior authorization required.",
            "Hepatitis B vaccination is covered for adults aged 60 years and older with risk factors for hepatitis B infection.",
        ]),
    },
    "0301": {
        "name": "CPB 0301 Knee Arthroscopy",
        "criteria": crit("0301", [
            "Arthroscopic lavage of the knee for osteoarthritis is not medically necessary.",
            "Arthroscopic meniscectomy is medically necessary for mechanical symptoms with a confirmed meniscal tear.",
        ]),
    },
    "0401": {
        "name": "CPB 0401 Hepatitis C Screening",
        "criteria": crit("0401", [
            "Hepatitis C antibody screening is covered once for adults aged 18 to 79 years.",
            "Repeat hepatitis C screening is covered annually for members who inject drugs.",
        ]),
    },
    "0501": {
        "name": "CPB 0501 Atorvastatin",
        "criteria": crit("0501", [
            "Atorvastatin is covered for adults with LDL cholesterol of 190 mg/dL or higher.",
        ]),
    },
}

GOLD_DETAIL = {
    ("G-CRC:1", "-"): "uncovered",           # 45-49 is outside every policy criterion
    ("G-CRC:2", "0101:1"): "aligned",
    ("G-CRC:3", "0101:3"): "policy_stricter",  # policy starts at 50, guideline at 45
    ("G-CRC:4", "0101:2"): "policy_stricter",
    ("G-CRC:5", "0101:4"): "conflict",       # selective screening 76-85 vs never necessary
    ("G-HBV:1", "0201:1"): "policy_stricter",  # prior authorization only in the policy
    ("G-HBV:2", "0201:2"): "aligned",
    ("G-LIP:1", "0501:1"): "policy_stricter",  # one statin vs the whole class
}
GOLD = {k: "consistent" if v == "aligned" else "diverges" for k, v in GOLD_DETAIL.items()}
GOLD_DIVERGING = {k for k, v in GOLD.items() if v == "diverges"}

_BOILER = (
    "Background. This bulletin describes the plan's position and is reviewed annually. "
    "Members and providers should verify benefits, which vary by plan; this document does "
    "not guarantee coverage. References: see the document's bibliography."
)


def render_plain(doc: dict) -> dict:
    """The 'plain text dump' shape: a name and one blob of prose, no structure."""
    body = " ".join(c["text"] for c in doc["criteria"])
    return {"document_name": doc["name"], "text": f"{doc['name']}\n\n{_BOILER}\n\nPosition. {body}"}


print(len(GUIDELINES), "guidelines,", len(POLICIES), "policies,", len(GOLD), "gold items,", len(GOLD_DIVERGING), "need review")

## 2. Scoring

All arms are reduced to `(guideline_criterion_id, policy_criterion_id | "-", verdict)`. Arm A cannot cite ids (it only sees prose), so its excerpts are mapped to the nearest criterion by token overlap for *scoring only*, which slightly flatters A.

In [ ]:
def prf(pred: set, gold: set) -> dict:
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0}


def score(pred: set[tuple[str, str, str]]) -> dict:
    pairs = {(g, p) for g, p, _ in pred}
    pair = prf(pairs, set(GOLD))
    found = {(g, p): v for g, p, v in pred}
    both = pairs & set(GOLD)
    verdict_acc = sum(found[k] == GOLD[k] for k in both) / len(both) if both else 0.0
    div = prf({(g, p) for g, p, v in pred if v == "diverges"}, GOLD_DIVERGING)
    return {
        "pair_precision": pair["precision"], "pair_recall": pair["recall"], "pair_f1": pair["f1"],
        "verdict_acc": verdict_acc, "diverge_f1": div["f1"],
    }


def ledger_summary() -> pd.DataFrame:
    df = pd.DataFrame(LEDGER)
    if df.empty:
        return df
    g = df.groupby("arm").agg(calls=("in", "size"), in_tokens=("in", "sum"), out_tokens=("out", "sum"))
    g["fresh_calls"] = df[~df.cached].groupby("arm").size()
    return g.fillna(0).astype(int)

## 3. Arm A: plain-text dump

One call per guideline. Context = the guideline as `{document_name, text}` plus **every** policy the same way. The model must locate and quote the passages itself.

In [ ]:
class TextMatch(BaseModel):
    guideline_excerpt: str
    policy_document_name: Optional[str]
    policy_excerpt: Optional[str]
    verdict: Literal["consistent", "diverges"]
    divergence: Optional[str]


class TextResult(BaseModel):
    matches: list[TextMatch]


TASK = (
    "You compare an external clinical guideline against payer coverage policies. Find every "
    "passage of a policy that addresses the same service, population and recommendation as a "
    "passage of the guideline. For each such pair, say whether the policy is `consistent` with "
    "the guideline or `diverges` from it (different age range or population, frequency, "
    "authorization requirement, or a conflicting coverage stance). Report only pairs that "
    "genuinely address the same clinical question and an overlapping population; do not match "
    "merely similar topics (e.g. a different disease). If no policy passage addresses a "
    "guideline passage, report it with null policy fields and verdict `diverges` (a policy gap). "
    "Quote excerpts verbatim."
)

_STOP = {"the", "a", "an", "of", "for", "to", "and", "or", "in", "with", "is", "are", "be", "by", "on", "as", "may", "should", "who"}


def toks(s: str) -> set[str]:
    out = set()
    for w in re.findall(r"[a-z0-9]+", s.lower()):
        if w in _STOP:
            continue
        for suf in ("ing", "ed", "es", "s"):  # crude stemmer, enough to join plural/gerund forms
            if w.endswith(suf) and len(w) - len(suf) >= 4:
                w = w[: -len(suf)]
                break
        out.add(w)
    return out


def nearest(excerpt: str, criteria: list[dict]) -> str:
    t = toks(excerpt)
    return max(criteria, key=lambda c: len(t & toks(c["text"])) / (len(t | toks(c["text"])) or 1))["criterion_id"]


def run_arm_a() -> set:
    plain_pol = {pn: render_plain(d) for pn, d in POLICIES.items()}
    by_name = {v["document_name"]: pn for pn, v in plain_pol.items()}
    pred = set()
    for g in GUIDELINES.values():
        payload = json.dumps({"guideline": render_plain(g), "policies": list(plain_pol.values())}, indent=1)
        for m in ask("A_plain_text", TextResult, TASK, payload).matches:
            gid = nearest(m.guideline_excerpt, g["criteria"])
            if m.policy_document_name is None or m.policy_excerpt is None:
                pred.add((gid, "-", m.verdict))
                continue
            hit = get_close_matches(m.policy_document_name, list(by_name), n=1, cutoff=0.6)
            if hit:  # an unknown document name is a hallucination: dropped
                pn = by_name[hit[0]]
                pred.add((gid, nearest(m.policy_excerpt, POLICIES[pn]["criteria"]), m.verdict))
    return pred

## 4. Arm B: atomized criteria dump

One call per guideline. Context = the guideline's criteria and **every** policy criterion as JSON (`criterion_id`, `parent_id`, `text`: what the current schema stores). The model can cite ids.

In [ ]:
class IdMatch(BaseModel):
    guideline_criterion_id: str
    policy_criterion_id: Optional[str]
    verdict: Literal["consistent", "diverges"]
    divergence: Optional[str]


class IdResult(BaseModel):
    matches: list[IdMatch]


def run_arm_b() -> set:
    all_policy_criteria = [{**c, "policy_title": p["name"]} for p in POLICIES.values() for c in p["criteria"]]
    valid_p = {c["criterion_id"] for c in all_policy_criteria}
    pred = set()
    for g in GUIDELINES.values():
        payload = json.dumps({"guideline_criteria": g["criteria"], "policy_criteria": all_policy_criteria}, indent=1)
        res = ask("B_criteria_dump", IdResult, TASK + " Cite criterion ids.", payload)
        valid_g = {c["criterion_id"] for c in g["criteria"]}
        pred |= {
            (m.guideline_criterion_id, m.policy_criterion_id or "-", m.verdict)
            for m in res.matches
            if m.guideline_criterion_id in valid_g and (m.policy_criterion_id is None or m.policy_criterion_id in valid_p)
        }
    return pred

## 5. Arm C: structured records, Concepts, rules

### 5a. Extract one record per criterion (LLM, offline, once per criterion)

In [ ]:
class Fact(BaseModel):
    concept: str                       # canonical name, e.g. "age", "type 2 diabetes mellitus"
    kind: Literal["condition", "measurement", "drug", "procedure"]
    present: bool                      # False = the patient does NOT have it / has not had it
    lo: Optional[float]                # inclusive lower bound for measurements (age in years, mg/dL ...)
    hi: Optional[float]


class Who(BaseModel):
    mode: Literal["all_of", "any_of"]
    facts: list[Fact]                  # empty = everyone


class Delivery(BaseModel):
    times_per_year: Optional[float]    # "once every 10 years" = 0.1, "annually" = 1
    setting: Optional[Literal["inpatient", "outpatient", "home", "office", "other"]]
    by_whom: Optional[str]
    prior_auth: Optional[bool]         # True only when authorization is stated as required


class Intervention(BaseModel):
    kind: Literal["drug", "drug_class", "vaccine", "procedure", "therapy", "test", "device", "other"]
    name: str                          # canonical English name of the thing itself


class Record(BaseModel):
    criterion_id: str
    stance: Literal["for", "against"]
    intervention: Intervention
    who: Who
    delivery: Delivery


class RecordBatch(BaseModel):
    records: list[Record]


EXTRACT_TASK = (
    "Read each criterion as one sentence: [stance] [intervention] for [who], limited by [delivery]. "
    "Return one record per criterion, echoing criterion_id.\n"
    "- stance: 'for' if it recommends, suggests, covers or calls the intervention medically "
    "necessary; 'against' if it recommends against it, does not cover it, or calls it "
    "experimental, investigational or not medically necessary.\n"
    "- intervention.name: the canonical English name of the thing done, as a standard textbook "
    "term, without purpose, population or setting words (a 'screening colonoscopy' is "
    "'colonoscopy'). Colloquial terms must be expanded ('colon scope' -> 'colonoscopy'). A group of "
    "drugs is kind 'drug_class' and uses the ATC class name (statins: 'HMG CoA reductase "
    "inhibitors'); a single drug uses its generic name.\n"
    "- who.facts: every statement about the patient. Age is concept 'age' (kind measurement, "
    "years; 'adults' = lo 18; 'older than 75' = lo 76; '45 to 49' = lo 45, hi 49). Lab values are "
    "measurements with a canonical name ('LDL cholesterol', mg/dL). Conditions are lowercase "
    "singular noun phrases; use the SAME wording for the same thing in every criterion ('average "
    "risk of colorectal cancer', 'risk factor for hepatitis B infection', 'pregnancy'). Prior "
    "treatments are kind 'drug' or 'procedure'. mode is 'any_of' only if the text lists "
    "alternatives; otherwise 'all_of'. Do not put anything in who that the text does not say.\n"
    "- delivery: only limits the text states; leave the rest null."
)


def extract_records() -> dict[str, Record]:
    out: dict[str, Record] = {}
    for d in [*GUIDELINES.values(), *POLICIES.values()]:
        res = ask("C_extract", RecordBatch, EXTRACT_TASK, json.dumps(d["criteria"], indent=1))
        out |= {r.criterion_id: r for r in res.records}
    missing = {c["criterion_id"] for d in [*GUIDELINES.values(), *POLICIES.values()] for c in d["criteria"]} - set(out)
    if missing:
        raise ValueError(f"extraction skipped {sorted(missing)}")
    return out

### 5b. Map terms to Concepts (lookup, not LLM)

`Normalizer` resolves each term against RxNorm / RxClass / SNOMED CT / CVX / LOINC / NCIt (keyless, see `intervention_normalizer.py`) and relates concepts (SNOMED subsumption, drug-in-class). A term no source resolves keeps a `text:` Concept, so identical wording still matches. `IdentityNormalizer` is the ablation: every Concept is just the lowercased words.

In [ ]:
INF = float("inf")
FACT_KIND = {"condition": Kind.CONDITION, "drug": Kind.DRUG, "procedure": Kind.PROCEDURE}


def _slug(s: str) -> str:
    return re.sub(r"\W+", "_", s.lower()).strip("_")


class IdentityNormalizer:
    """Ablation: no vocabularies. Two terms are related only if the words are identical."""

    def normalize(self, kind: Kind, text: str) -> Normalized:
        return Normalized(text, kind, Concept("text", _slug(text), text))

    def related(self, a: Optional[Concept], b: Optional[Concept]) -> str:
        return "equal" if a is not None and b is not None and a.key == b.key else "unrelated"


@dataclass(frozen=True)
class Atom:
    concept: Concept
    present: bool
    lo: float
    hi: float


@dataclass(frozen=True)
class Rec:
    criterion_id: str
    stance: str
    concept: Concept                       # the intervention
    dnf: tuple[tuple[Atom, ...], ...]      # who as OR of ANDs; ((),) = everyone
    delivery: Delivery


NORM_LOG: dict[tuple[str, str, str], Concept] = {}  # (normalizer, kind, text) -> concept, for the report


def concept_for(norm, label: str, kind: Kind, text: str) -> Concept:
    concept = norm.normalize(kind, text).concept or Concept("text", _slug(text), text)
    NORM_LOG[(label, kind.value, text.lower())] = concept
    return concept


def build_rec(r: Record, norm, label: str) -> Rec:
    atoms = []
    for f in r.who.facts:
        if f.kind == "measurement":
            c = Concept("measure", _slug(f.concept), f.concept)
        else:
            c = concept_for(norm, label, FACT_KIND[f.kind], f.concept)
        atoms.append(Atom(c, f.present, -INF if f.lo is None else f.lo, INF if f.hi is None else f.hi))
    dnf = (tuple(atoms),) if r.who.mode == "all_of" else tuple((a,) for a in atoms)
    return Rec(r.criterion_id, r.stance, concept_for(norm, label, Kind(r.intervention.kind), r.intervention.name), dnf or ((),), r.delivery)

### 5c. Compare slot by slot (fixed rules, no LLM)

`who` is a set of patients in disjunctive normal form (`any_of` = OR of single facts, `all_of` = one AND). Subset / superset / disjoint follow from interval arithmetic plus Concept subsumption.

In [ ]:
def implies(a: Atom, b: Atom, norm) -> bool:
    """Every patient matching atom a also matches atom b."""
    if a.present != b.present:
        return False
    rel = norm.related(a.concept, b.concept)
    concept_ok = rel in ("equal", "a_narrower") if a.present else rel == "equal"
    return concept_ok and b.lo <= a.lo and a.hi <= b.hi


def contradicts(a: Atom, b: Atom, norm) -> bool:
    """No patient can match both atoms."""
    if norm.related(a.concept, b.concept) != "equal":
        return False
    return a.present != b.present or (a.present and (a.hi < b.lo or b.hi < a.lo))


def conj_implies(a: tuple, b: tuple, norm) -> bool:
    return all(any(implies(x, y, norm) for x in a) for y in b)


def who_relation(policy: tuple, guideline: tuple, norm) -> str:
    """Policy's patient set relative to the guideline's: equal, subset, superset, disjoint, overlap."""
    subset = all(any(conj_implies(a, b, norm) for b in guideline) for a in policy)
    superset = all(any(conj_implies(b, a, norm) for a in policy) for b in guideline)
    if subset and superset:
        return "equal"
    if subset:
        return "subset"
    if superset:
        return "superset"
    if all(any(contradicts(x, y, norm) for x in a for y in b) for a in policy for b in guideline):
        return "disjoint"
    return "overlap"


@dataclass(frozen=True)
class Verdict:
    category: str            # aligned, policy_stricter, policy_looser, mixed, conflict, uncovered
    reasons: tuple[str, ...]


def compare_delivery(g: Delivery, p: Delivery) -> tuple[list[str], list[str]]:
    stricter, looser = [], []
    if g.times_per_year is None and p.times_per_year is not None:
        stricter.append("policy limits frequency, guideline does not")
    elif p.times_per_year is None and g.times_per_year is not None:
        looser.append("guideline limits frequency, policy does not")
    elif g.times_per_year is not None and p.times_per_year is not None and g.times_per_year != p.times_per_year:
        (stricter if p.times_per_year < g.times_per_year else looser).append(
            f"frequency {p.times_per_year}/yr in policy vs {g.times_per_year}/yr in guideline")
    for slot in ("setting", "by_whom"):
        gv, pv = getattr(g, slot), getattr(p, slot)
        if gv is None and pv is not None:
            stricter.append(f"policy restricts {slot}: {pv}")
        elif pv is None and gv is not None:
            looser.append(f"guideline restricts {slot}: {gv}")
        elif gv is not None and pv is not None and str(gv).lower() != str(pv).lower():
            stricter.append(f"{slot} differs: policy {pv}, guideline {gv}")
            looser.append(f"{slot} differs: policy {pv}, guideline {gv}")
    if p.prior_auth and not g.prior_auth:
        stricter.append("policy requires prior authorization")
    elif g.prior_auth and not p.prior_auth:
        looser.append("guideline requires prior authorization, policy does not")
    return stricter, looser


def compare(g: Rec, p: Rec, norm) -> Optional[Verdict]:
    """None = the pair is not decidable or not about the same patients."""
    who = who_relation(p.dnf, g.dnf, norm)
    if who == "disjoint":
        return None
    if g.stance != p.stance:
        return Verdict("conflict", (f"guideline is {g.stance}, policy is {p.stance}",))
    stricter, looser = [], []
    if who == "subset":
        stricter.append("policy applies to a narrower patient group")
    elif who == "superset":
        looser.append("policy applies to a wider patient group")
    elif who == "overlap":
        return None  # same stance, incomparable patient sets: not guessed
    rel = norm.related(g.concept, p.concept)  # guideline relative to policy
    if rel == "a_broader":
        stricter.append(f"policy names a narrower intervention ({p.concept.name})")
    elif rel == "a_narrower":
        looser.append(f"policy names a broader intervention ({p.concept.name})")
    ds, dl = compare_delivery(g.delivery, p.delivery)
    stricter += ds
    looser += dl
    if stricter and looser:
        return Verdict("mixed", tuple(stricter + looser))
    if stricter:
        return Verdict("policy_stricter", tuple(stricter))
    if looser:
        return Verdict("policy_looser", tuple(looser))
    return Verdict("aligned", ())


def run_arm_c(records: dict[str, Record], norm, label: str):
    g_recs = [build_rec(records[c["criterion_id"]], norm, label) for g in GUIDELINES.values() for c in g["criteria"]]
    p_recs = [build_rec(records[c["criterion_id"]], norm, label) for p in POLICIES.values() for c in p["criteria"]]
    pred, detail = set(), {}
    for g in g_recs:
        hits = []
        for p in p_recs:
            if norm.related(g.concept, p.concept) != "unrelated":   # same, broader or narrower intervention
                v = compare(g, p, norm)
                if v:
                    hits.append((p.criterion_id, v))
        if not hits:
            hits = [("-", Verdict("uncovered", ("no policy criterion addresses this",)))]
        for pid, v in hits:
            pred.add((g.criterion_id, pid, "consistent" if v.category == "aligned" else "diverges"))
            detail[(g.criterion_id, pid)] = v
    return pred, detail

### 5d. The LLM writes the sentence (online, only for pairs that need review)

The verdict and the exact difference are already decided; the model only turns them into prose. It cannot change them.

In [ ]:
class Explanation(BaseModel):
    guideline_criterion_id: str
    policy_criterion_id: str
    sentence: str


class ExplanationBatch(BaseModel):
    items: list[Explanation]


EXPLAIN_TASK = (
    "For each item, write one plain sentence a policy reviewer can act on, stating how the payer "
    "policy criterion differs from the guideline criterion. Use only the given category and reasons; "
    "do not add facts or change the verdict. Echo both ids."
)


def explain(detail: dict) -> list[Explanation]:
    text = {c["criterion_id"]: c["text"] for d in [*GUIDELINES.values(), *POLICIES.values()] for c in d["criteria"]}
    items = [
        {"guideline_criterion_id": g, "policy_criterion_id": p, "guideline": text[g],
         "policy": text.get(p), "category": v.category, "reasons": list(v.reasons)}
        for (g, p), v in sorted(detail.items()) if v.category != "aligned"
    ]
    return ask("C_explain", ExplanationBatch, EXPLAIN_TASK, json.dumps(items, indent=1)).items if items else []

## 6. Run all arms

In [ ]:
LEDGER.clear()
NORM_LOG.clear()
records = extract_records()                                   # LLM, offline
norm = Normalizer(cache_path=CACHE_DIR / "normalizer.json")   # public keyless vocabularies

preds = {"A_plain_text": run_arm_a(), "B_criteria_dump": run_arm_b()}
details = {}
preds["C_ontology"], details["C_ontology"] = run_arm_c(records, norm, "norm")
preds["C_no_normalization"], details["C_no_normalization"] = run_arm_c(records, IdentityNormalizer(), "ident")

results = pd.DataFrame({arm: score(p) for arm, p in preds.items()}).T
results["category_acc"] = [
    None if arm not in details else sum(details[arm].get(k) is not None and details[arm][k].category == v for k, v in GOLD_DETAIL.items()) / len(GOLD_DETAIL)
    for arm in results.index
]
results.round(2)

In [ ]:
# Cost. C_extract is offline (once per criterion, reused for every future guideline).
# Only C_explain is paid per matching run, and only for pairs that need review.
explanations = explain(details["C_ontology"])
cost = ledger_summary()
n_all = sum(len(p["criteria"]) for p in POLICIES.values())
print(f"policy criteria the LLM must read per guideline: A/B = all {n_all}; C = 0 (code selects)")
cost

## 7. Where each arm went wrong

In [ ]:
rows = []
for arm, pred in preds.items():
    found = {(g, p): v for g, p, v in pred}
    for k in sorted(set(found) | set(GOLD)):
        if found.get(k) != GOLD.get(k):
            kind = "missed" if k not in found else "spurious" if k not in GOLD else "wrong verdict"
            rows.append({"arm": arm, "guideline_crit": k[0], "policy_crit": k[1], "error": kind, "got": found.get(k), "gold": GOLD.get(k)})
errors = pd.DataFrame(rows)
errors if len(errors) else "no errors"

In [ ]:
# Arm C's verdicts next to the gold category, with the sentence the LLM wrote for each.
said = {(e.guideline_criterion_id, e.policy_criterion_id): e.sentence for e in explanations}
pd.DataFrame([
    {"guideline": g, "policy": p, "got": v.category, "gold": GOLD_DETAIL.get((g, p)), "reasons": "; ".join(v.reasons), "sentence": said.get((g, p))}
    for (g, p), v in sorted(details["C_ontology"].items())
])

## 8. What normalization did

Every term mapped to a Concept, with what each vocabulary returned. Rows whose `system` is `text` were not resolved by any source and fall back to the wording itself; if the same real-world thing appears there under two spellings, matching will miss it. This is the table to read before trusting arm C.

In [ ]:
pd.DataFrame(
    [{"kind": k, "term": t, "system": c.system, "code": c.code, "concept": c.name}
     for (label, k, t), c in sorted(NORM_LOG.items()) if label == "norm"]
)

## 9. Next steps if the signal is promising

- Replace the fixtures with real `guideline_criterion` / `policy_criterion` rows and a labelled sample from `needs_update` reviews.
- Run each arm several times (extraction and arms A/B are stochastic); at this sample size single runs are noise.
- Upgrade the Concept layer: a free **UMLS licence** adds cross-vocabulary synonyms and crosswalks, and the hospital's IMO / Health Language / CPT access covers colloquial terms and billing codes. All options are listed in `docs/INTERVENTION_NORMALIZATION.md`.
- Store the record and Concepts in the DB (`facets` jsonb plus a code table) so matching becomes a SQL join on Concept.